<a href="https://colab.research.google.com/github/jaya-adurthi/Code-Buddy/blob/main/CodeBuddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q streamlit mistralai==0.4.2

In [3]:
import os
from google.colab import userdata
os.environ['MISTRAL_API_KEY']=userdata.get('MISTRAL_API_KEY')

In [4]:
%%writefile app.py

# ============================================================
# 💻 Code Buddy — AI Coding Assistant for Students
# Streamlit + Mistral AI (old SDK v0.x)
# Author: Bapuji Kanaparthi
# ============================================================

import os
import streamlit as st
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage

api_key = os.environ.get("MISTRAL_API_KEY")
client = MistralClient(api_key=api_key)

st.set_page_config(page_title="Code Buddy", page_icon="💻", layout="wide")
st.title("💻 Code Buddy")
st.caption("Your AI pair programmer — write, explain, debug, and optimize code")

# ---------------- Modes ----------------
MODES = {
    "✍️ Write Code": "You are an expert programmer. Write clean, well-commented, working code for the user's request. Always use proper markdown code blocks with the language tag. Briefly explain the approach after the code.",
    "🔍 Explain Code": "You are a patient programming teacher for engineering students. Explain the given code step by step in simple language. Use analogies where helpful. Point out any important concepts used.",
    "🐛 Debug Code": "You are a debugging expert. Find the bugs in the given code, explain WHY each bug happens, then provide the corrected code in a markdown code block. List the fixes clearly.",
    "⚡ Optimize Code": "You are a performance expert. Analyze the given code, suggest optimizations (time complexity, memory, readability), and provide the improved version with comments explaining each change.",
    "🔄 Convert Language": "You are a polyglot programmer. Convert the given code to the requested target language, keeping the same logic. Note any language-specific differences.",
    "📝 Add Comments": "You are a documentation expert. Add clear docstrings and comments to the given code without changing its logic. Return the fully commented code.",
    "🎯 Interview Prep": "You are a coding interview coach. Give the user a coding problem based on their topic, wait for their solution, then review it like an interviewer — correctness, complexity, edge cases — and rate it out of 10.",
}

LANGUAGES = ["Python", "C", "C++", "Java", "JavaScript", "SQL", "Embedded C (Arduino)", "Verilog"]

# ---------------- Sidebar ----------------
with st.sidebar:
    mode = st.selectbox("Mode", list(MODES.keys()))
    language = st.selectbox("Preferred Language", LANGUAGES)
    model = st.selectbox("Model", ["mistral-large-latest", "mistral-small-latest"])
    st.divider()
    st.markdown("**Quick Tasks**")
    quick = None
    if st.button("Binary Search", use_container_width=True):
        quick = f"Write binary search in {language} with comments"
    if st.button("Linked List", use_container_width=True):
        quick = f"Implement a singly linked list in {language}"
    if st.button("LED Blink (IoT)", use_container_width=True):
        quick = "Write Arduino code to blink an LED with a button interrupt"
    if st.button("Star Pattern", use_container_width=True):
        quick = f"Print a pyramid star pattern in {language}"
    st.divider()
    if st.button("🗑 Clear Chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

if "messages" not in st.session_state:
    st.session_state.messages = []

# ---------------- Code Paste Box ----------------
with st.expander("📋 Paste your code here (optional — for Explain / Debug / Optimize)"):
    user_code = st.text_area("Code", height=200, label_visibility="collapsed")

# ---------------- Show History ----------------
for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

# ---------------- Chat ----------------
prompt = st.chat_input("Ask a coding question or describe what to do with your code...")

if not prompt and quick:
    prompt = quick

if prompt:
    # Attach pasted code if present
    full_prompt = prompt
    if user_code.strip():
        full_prompt = f"{prompt}\n\n```\n{user_code}\n```"

    st.session_state.messages.append({"role": "user", "content": full_prompt})
    with st.chat_message("user"):
        st.markdown(full_prompt)

    system = MODES[mode] + f" The student's preferred language is {language}."
    history = [ChatMessage(role="system", content=system)]
    for m in st.session_state.messages[-10:]:
        history.append(ChatMessage(role=m["role"], content=m["content"]))

    with st.chat_message("assistant"):
        placeholder = st.empty()
        answer = ""
        try:
            stream = client.chat_stream(
                model=model,
                messages=history,
                temperature=0.3,
            )
            for chunk in stream:
                token = chunk.choices[0].delta.content
                if token:
                    answer += token
                    placeholder.markdown(answer + "▌")
            placeholder.markdown(answer)
        except Exception as e:
            answer = f"❌ Error: {e}"
            placeholder.error(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})

Writing app.py


In [5]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [6]:
import subprocess, time, requests, os, signal

# clean slate
os.system("pkill -f streamlit")
time.sleep(2)

proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.fileWatcherType", "none",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)

ok = False
for i in range(60):
    if proc.poll() is not None:          # process exited
        print(f"❌ Streamlit exited with code {proc.returncode}. Log:")
        print(open("streamlit.log").read())
        break
    try:
        requests.get("http://localhost:8501", timeout=2)
        ok = True
        print("✅ Streamlit is running and responding")
        break
    except Exception:
        time.sleep(2)

if not ok and proc.poll() is None:
    print("⚠️ Still not responding after 2 min. Log so far:")
    print(open("streamlit.log").read())

✅ Streamlit is running and responding


In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-07-26T06:03:15Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-26T06:03:15Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-26T06:03:19Z INF +--------------------------------------------------------------------------------------------+
2026-07-26T06:03:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-26T06:03:19Z INF |  https://day-spotlight-attend-bills.trycloudflare.com 